In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import cv2
import numpy as np
from pathlib import Path
from typing import Literal

PostprocessType = Literal[
    "gaussian_blur",
    "resize",
    "color_saturation",
    "color_contrast",
]

BLUR_KERNELS = {
    1: 5,
    2: 9,
    3: 13,
    4: 17,
    5: 21,
}

RESIZE_INTERMEDIATE = {
    1: 128,
    2: 85,
    3: 64,
    4: 51,
    5: 41,
}

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _validate_level(level: int) -> None:
    if level not in {1, 2, 3, 4, 5}:
        raise ValueError("level must be one of {1, 2, 3, 4, 5}")


def _validate_levels(levels: list[int]) -> None:
    if not levels:
        raise ValueError("levels must not be empty")
    for level in levels:
        _validate_level(level)


def gaussian_blur(img_bgr: np.ndarray, level: int) -> np.ndarray:
    _validate_level(level)
    k = BLUR_KERNELS[level]
    return cv2.GaussianBlur(img_bgr, (k, k), sigmaX=0)


def resize_degrade(
    img_bgr: np.ndarray,
    level: int,
    output_size: tuple[int, int] = (256, 256),
) -> np.ndarray:
    _validate_level(level)
    inter = RESIZE_INTERMEDIATE[level]
    small = cv2.resize(img_bgr, (inter, inter), interpolation=cv2.INTER_AREA)
    restored = cv2.resize(small, output_size, interpolation=cv2.INTER_LINEAR)
    return restored


def color_saturation(img_bgr: np.ndarray, level: int) -> np.ndarray:
    _validate_level(level)

    img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb).astype(np.float32)

    y = img_ycrcb[:, :, 0]
    cr = img_ycrcb[:, :, 1]
    cb = img_ycrcb[:, :, 2]

    factor = float(level)
    cr = 128.0 + (cr - 128.0) * factor
    cb = 128.0 + (cb - 128.0) * factor

    out = np.stack([y, cr, cb], axis=-1)
    out = np.clip(out, 0, 255).astype(np.uint8)
    return cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)


def color_contrast(img_bgr: np.ndarray, level: int) -> np.ndarray:
    _validate_level(level)

    img = img_bgr.astype(np.float32)
    mean = img.mean(axis=(0, 1), keepdims=True)
    factor = float(level)

    out = mean + (img - mean) * factor
    out = np.clip(out, 0, 255).astype(np.uint8)
    return out


def apply_postprocessing(
    img_bgr: np.ndarray,
    method: PostprocessType,
    level: int,
    output_size: tuple[int, int] = (256, 256),
) -> np.ndarray:
    if method == "gaussian_blur":
        return gaussian_blur(img_bgr, level)
    if method == "resize":
        return resize_degrade(img_bgr, level, output_size=output_size)
    if method == "color_saturation":
        return color_saturation(img_bgr, level)
    if method == "color_contrast":
        return color_contrast(img_bgr, level)

    raise ValueError(f"Unknown method: {method}")


def iter_image_files(input_dir: Path) -> list[Path]:
    return sorted(
        p for p in input_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
    )


def process_folder(
    input_dir: str,
    output_root: str,
    dataset_name: str,
    methods: list[PostprocessType] | None = None,
    output_size: tuple[int, int] = (256, 256),
    levels: list[int] | None = None
) -> None:
    input_path = Path(input_dir)
    output_path = Path(output_root)

    if not input_path.exists() or not input_path.is_dir():
        raise ValueError(f"Input folder does not exist or is not a directory: {input_dir}")

    output_path.mkdir(parents=True, exist_ok=True)

    if methods is None:
        methods = [
            "gaussian_blur",
            "resize",
            "color_saturation",
            "color_contrast",
        ]

    if levels is None:
        levels = [1, 2, 3, 4, 5]

    _validate_levels(levels)

    image_files = iter_image_files(input_path)
    if not image_files:
        raise ValueError(f"No image files found in: {input_dir}")

    for img_file in image_files:
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"[WARN] Cannot read image: {img_file}")
            continue

        img = cv2.resize(img, output_size, interpolation=cv2.INTER_LINEAR)

        # giữ cấu trúc folder
        rel_path = img_file.relative_to(input_path)

        for method in methods:
            for level in levels:
                out = apply_postprocessing(
                    img_bgr=img,
                    method=method,
                    level=level,
                    output_size=output_size,
                )

                save_path = (
                    output_path
                    / method
                    / f"level_{level}"
                    / dataset_name
                    / rel_path
                )

                # tạo folder cha (bao gồm class folder)
                save_path.parent.mkdir(parents=True, exist_ok=True)

                success = cv2.imwrite(str(save_path), out)
                if not success:
                    print(f"[WARN] Failed to save image: {save_path}")

    print(f"Done. Results saved to: {output_root}")


if __name__ == "__main__":
    input_dir = "/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench/Celeb-DF-v1"
    output_root = "/kaggle/working/processed_output"

    process_folder(
        input_dir=input_dir,
        output_root=output_root,
        dataset_name = "Celeb-DF-v1",
        methods=[
            # "gaussian_blur",
            # "resize",
            "color_saturation",
            # "color_contrast",
        ],
        levels=[5],  # chọn level cần chạy
        output_size=(256, 256),
    )

Done. Results saved to: /kaggle/working/processed_output
